In [2]:
import os
import json
import asyncio
from typing import TypedDict, List, Dict
from neo4j import GraphDatabase
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()
# AutoGen Native Chat imports
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo



In [3]:
import nest_asyncio

# Re-engineers Python's internal loop tracker to allow nesting
nest_asyncio.apply() 


In [4]:

# --- 1. CONFIGURATION STACK ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "MySecretGraph2026!"

# --- 2. NEO4J PERSISTENCE HANDLER ---
def save_to_neo4j(triples: List[Dict[str, str]]):
    """Safely updates entities and edges inside your graph database container."""
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        with driver.session() as session:
            for t in triples:
                if not all(k in t for k in ('subject', 'predicate', 'object')):
                    continue # Skip structural anomalies
                
                rel_type = t['predicate'].strip().upper().replace(" ", "_")
                sub = t['subject'].strip().title()
                obj = t['object'].strip().title()
                
                session.run(
                    f"MERGE (s:Entity {{name: $sub}}) "
                    f"MERGE (o:Entity {{name: $obj}}) "
                    f"MERGE (s)-[:{rel_type}]->(o)", 
                    sub=sub, obj=obj
                )


In [5]:

# --- 3. LANGGRAPH STATE DEFINITION ---
class PipelineState(TypedDict):
    raw_text: str
    extracted_triples: List[Dict[str, str]]
    target_search: Dict[str, str]
    reasoning_result: str
    
class Triple(BaseModel):
    subject: str = Field(description="The main entity or noun of the fact.")
    predicate: str = Field(description="The relationship or action verb linking the subject and object.")
    object: str = Field(description="The target entity, concept, or value.")

class KnowledgeGraph(BaseModel):
    triples: List[Triple] = Field(description="A list of all extracted factual triples.")



In [10]:
# --- 4. AGENTIC AGENT WORKFLOW ENGINE ---
async def extract_and_validate_with_autogen(text: str) -> List[Dict[str, str]]:
    """Runs a multi-agent debate to fetch high-integrity triples using Groq."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        print("❌ Missing GROQ_API_KEY. Defaulting to empty graph ingestion.")
        return []

    
    groq_llm = OpenAIChatCompletionClient(
       model="qwen/qwen3.6-27b",
           api_key=api_key,
           base_url="https://api.groq.com/openai/v1",
           reasoning_effort="none",
           response_format={"type": "json_object"},
           model_info=ModelInfo(
               vision=False,
               function_calling=True,
               json_output=True,
               family="unknown"
           )
    )
    groq_llm_text = OpenAIChatCompletionClient(
       model="qwen/qwen3.6-27b",
           api_key=api_key,
           base_url="https://api.groq.com/openai/v1",
           reasoning_effort="none",
           model_info=ModelInfo(
               vision=False,
               function_calling=True,
               json_output=True,
               family="unknown"
           )
    )


    miner = AssistantAgent(
        name="Data_Miner",
        model_client=groq_llm,
        system_message=(
        "You are a knowledge graph extraction agent. "
        "Return ONLY valid JSON. "
        "Do not output markdown. "
        "Do not provide reasoning or explanations."
        "Do not output <think> tags. "
        "Your response must be a JSON object with a 'triples' array. "
        "Each triple must contain 'subject', 'predicate', and 'object'. "
        'Example: {"triples":[{"subject":"A","predicate":"related_to","object":"B"}]}'

        )
    )


    critic = AssistantAgent(
        name="Critic",
        model_client=groq_llm_text,
        system_message=(
            "Check the Data_Miner's structured list. Ensure all relationships are truthful to the text. "
            "Do not output <think> tags. "
            "If the output matches the text facts accurately, reply exactly with: APPROVED. "
            "If wrong, explain what needs correction."
        )
    )

    team = RoundRobinGroupChat([miner, critic], max_turns=4)
    task = f"Extract graph data accurately from this text:\n{text}"
    
    final_json_string = "[]"
    
    async for message in team.run_stream(task=task):
        if hasattr(message, "content") is False:
            continue
        
        content = message.content or ""
        print(f"Content : {content}")
        
        
        if message.source == "Data_Miner" and content.strip().startswith("{"):
            final_json_string = content.strip()
        
        if message.source == "Critic" and "APPROVED" in content.upper():
            print("🎯 [Team Orchestrator] Critic approved the data. Terminating loop early!")
            break

    try:
        # Load the structured payload and drill down into the 'triples' key array
        structured_data = json.loads(final_json_string)
        return structured_data.get("triples", [])
    except Exception as e:
        print(f"⚠️ JSON extraction parse anomaly handled: {e}")
        return []


In [7]:

# --- 5. ORCHESTRATION PIPELINE NODES ---
def multi_agent_extraction_node(state: PipelineState) -> Dict:
    print("🤖 [Node: Extraction] Spinning up AutoGen Multi-Agent Team (Groq)...")
    loop = asyncio.get_event_loop()
    triples = loop.run_until_complete(extract_and_validate_with_autogen(state["raw_text"]))
    print(f"📈 Found {len(triples)} high-fidelity triples.")
    return {"extracted_triples": triples}

def storage_persistence_node(state: PipelineState) -> Dict:
    print(f"🗄️ [Node: Storage] Injecting approved triples into Neo4j indices...")
    save_to_neo4j(state["extracted_triples"])
    return {}

def relational_reasoning_node(state: PipelineState) -> Dict:
    print("🧠 [Node: Reasoning] Invoking lightning-fast Cypher shortest path execution...")
    start = state['target_search'].get('start').title()
    end = state['target_search'].get('end').title()
    
    cypher_query = """
    MATCH path = shortestPath((s:Entity {name: $start})-[*..5]->(e:Entity {name: $end}))
    RETURN path
    """
    
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        with driver.session() as session:
            result = session.run(cypher_query, start=start, end=end)
            record = result.single()
            if record:
                nodes = [node['name'] for node in record['path'].nodes]
                return {"reasoning_result": f"✅ Multi-Agent Production Proof: " + " -> ".join(nodes)}
            
    return {"reasoning_result": f"❌ Could not infer connections from {start} to {end}."}


In [11]:

# --- 6. COMPILE SYSTEM RUNTIME ---
workflow = StateGraph(PipelineState)

workflow.add_node("autogen_extraction", multi_agent_extraction_node)
workflow.add_node("neo4j_storage", storage_persistence_node)
workflow.add_node("cypher_reasoner", relational_reasoning_node)

workflow.set_entry_point("autogen_extraction")
workflow.add_edge("autogen_extraction", "neo4j_storage")
workflow.add_edge("neo4j_storage", "cypher_reasoner")
workflow.add_edge("cypher_reasoner", END)

production_engine = workflow.compile()


In [13]:
sample_corpus = """
    Autogen is an AI framework engineered by Microsoft.
    Microsoft is headquartered in Redmond, Washington.
    Redmond is a city in Washington state.
    Washington is part of the United States.
    """
    
payload = {
    "raw_text": sample_corpus,
    "target_search": {"start": "Autogen", "end": "United States"}
}

print("🚀 Running Integrated Project 10 Core Engine...")
result = production_engine.invoke(payload)

print("\n🏁 [Final Execution Insight]:")
print(result["reasoning_result"])


🚀 Running Integrated Project 10 Core Engine...
🤖 [Node: Extraction] Spinning up AutoGen Multi-Agent Team (Groq)...
Content : Extract graph data accurately from this text:

    Autogen is an AI framework engineered by Microsoft.
    Microsoft is headquartered in Redmond, Washington.
    Redmond is a city in Washington state.
    Washington is part of the United States.
    
Content : {"triples":[{"subject":"Autogen","predicate":"engineered_by","object":"Microsoft"},{"subject":"Microsoft","predicate":"headquartered_in","object":"Redmond"},{"subject":"Microsoft","predicate":"headquartered_in","object":"Washington"},{"subject":"Redmond","predicate":"is_a","object":"city"},{"subject":"Redmond","predicate":"located_in","object":"Washington"},{"subject":"Washington","predicate":"is_part_of","object":"United States"}]}
Content : The extracted triples are mostly accurate, but there is one subtle implication issue. The first sentence explicitly states "Autogen is an AI framework engineered by Mi